# 🚧 Pothole INSTANCE SEGMENTATION — YOLOv11 (Google Colab)
### ⚠️ BEFORE YOU START:
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Run **Step 1 ONLY**, then **Runtime → Restart Session**
3. Run **Steps 2, 3, 4, 5** in order

In [ ]:
# ══════════════════════════════════════════════════════════
# STEP 0: Keep-Alive (Run this FIRST before anything else!)
# ▶ Prevents Colab from disconnecting during training!
# ══════════════════════════════════════════════════════════

In [ ]:
%%javascript
function ClickConnect(){
    console.log('Keeping Colab alive...');
    document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(ClickConnect, 60000)

In [ ]:
# ══════════════════════════════════════════════════════════
# STEP 1: Install Dependencies
# ▶ Run this cell. Then go to Runtime → Restart Session.
# ══════════════════════════════════════════════════════════
!pip install ultralytics roboflow -q
print('\n✅ Done! Now go to Runtime → Restart Session, then run Steps 2-5.')

In [ ]:
# ══════════════════════════════════════════════════════════
# STEP 2: GPU Check
# ▶ Run AFTER restarting. Must show CUDA = True.
# ══════════════════════════════════════════════════════════
import torch
print('=' * 50)
print(f'  PyTorch version : {torch.__version__}')
print(f'  CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU             : {torch.cuda.get_device_name(0)}')
    print('  ✅ Ready to train!')
else:
    print('  ❌ No GPU! Go to Runtime → Disconnect and delete runtime')
    print('     Then Runtime → Change runtime type → T4 GPU → Reconnect')
print('=' * 50)

In [ ]:
# ══════════════════════════════════════════════════════════
# STEP 3: Mount Google Drive
# ══════════════════════════════════════════════════════════
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/RoadSafe_Segmentation'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'✅ Google Drive mounted! Model auto-saves to: {DRIVE_DIR}')

In [ ]:
# ══════════════════════════════════════════════════════════
# STEP 4: Download Segmentation Dataset
# ══════════════════════════════════════════════════════════
from roboflow import Roboflow
rf = Roboflow(api_key="XgnrdkrD2jrcSAiJoiHY")
project = rf.workspace("gp-grakz").project("pothole-segmentation-g6hbh")
version = project.version(14)
dataset = version.download("yolov8")
print('✅ Dataset downloaded!')

In [ ]:
# ══════════════════════════════════════════════════════════
# STEP 5: Train YOLOv11 Segmentation — 200 Epochs
# ▶ ⏱️ Takes 3-4 hours. Do NOT close the tab!
# ▶ ✅ best.pt is auto-saved to Drive after EVERY epoch!
# ══════════════════════════════════════════════════════════
from ultralytics import YOLO
import shutil, glob, os

DRIVE_DIR = '/content/drive/MyDrive/RoadSafe_Segmentation'

# Auto-find the data.yaml
yaml_files = glob.glob('/content/**/data.yaml', recursive=True)
if not yaml_files:
    raise ValueError('❌ data.yaml not found! Run Step 4 first.')
yaml_path = yaml_files[0]
print(f'✅ Using dataset: {yaml_path}')

# Load model
model = YOLO('yolo11m-seg.pt')

# ✅ Auto-save best.pt to Google Drive after every epoch
def save_best_to_drive(trainer):
    best = f'{trainer.save_dir}/weights/best.pt'
    if os.path.exists(best):
        shutil.copy2(best, f'{DRIVE_DIR}/best_seg.pt')
        print(f'💾 Saved best.pt to Drive at Epoch {trainer.epoch + 1}')

model.add_callback('on_fit_epoch_end', save_best_to_drive)

results = model.train(
    data         = yaml_path,
    epochs       = 200,
    imgsz        = 640,
    batch        = 16,
    optimizer    = 'AdamW',
    lr0          = 0.001,
    lrf          = 0.01,
    patience     = 40,
    save         = True,
    project      = '/content/runs',
    name         = 'pothole_seg_yolo11',
    device       = 0,
    hsv_h        = 0.015,
    hsv_s        = 0.6,
    hsv_v        = 0.5,
    fliplr       = 0.5,
    mosaic       = 1.0,
    mixup        = 0.1,
    scale        = 0.5,
    translate    = 0.1,
)

print('\n✅ Training complete!')
shutil.copy2(f'{results.save_dir}/weights/best.pt', f'{DRIVE_DIR}/best_seg_final.pt')
print(f'🎉 Final model saved to Drive: {DRIVE_DIR}/best_seg_final.pt')